# 📊 ROC Curve & AUC — Complete Study Notebook

---

## 🎯 What You Will Learn

| Topic | Description |
|---|---|
| ROC Curve | What it is, how it is built, how to read it |
| Thresholds | How classification cut-offs affect TPR and FPR |
| AUC Score | How to quantify overall model quality |
| Optimal Threshold | How to find the best operating point |
| Model Comparison | Using ROC to pick between classifiers |

---

## 🧠 Background: The Classification Problem

A **binary classifier** outputs a probability (0–1) for each sample. We then apply a **threshold** (commonly 0.5) to convert that probability into a class label (0 or 1).

### Confusion Matrix Refresher

```
                   Predicted Positive   Predicted Negative
Actual Positive  |  TP (True Pos)    |  FN (False Neg)   |
Actual Negative  |  FP (False Pos)   |  TN (True Neg)    |
```

Key rates derived from this:

- **TPR (True Positive Rate) = Sensitivity = Recall** = TP / (TP + FN)
  - *"Of all actual positives, what fraction did we catch?"*
- **FPR (False Positive Rate) = 1 - Specificity** = FP / (FP + TN)
  - *"Of all actual negatives, what fraction did we wrongly flag?"*

---

## 📈 What is the ROC Curve?

The **Receiver Operating Characteristic (ROC)** curve plots **TPR vs FPR** as the classification threshold varies from 1.0 down to 0.0.

- **Threshold = 1.0** → model predicts everything as Negative → TPR=0, FPR=0 (bottom-left corner)
- **Threshold = 0.0** → model predicts everything as Positive → TPR=1, FPR=1 (top-right corner)
- **A perfect model** hugs the top-left corner: high TPR, zero FPR
- **A random model** falls on the diagonal line y = x

---

## 📐 What is AUC?

**AUC (Area Under the ROC Curve)** summarises the entire curve into one number:

| AUC Value | Interpretation |
|---|---|
| 1.00 | Perfect classifier |
| 0.90–0.99 | Excellent |
| 0.80–0.89 | Good |
| 0.70–0.79 | Fair |
| 0.60–0.69 | Poor |
| 0.50 | No better than random guessing |
| < 0.50 | Worse than random (predictions are inverted) |

> **Probabilistic interpretation:** AUC = the probability that a randomly chosen positive sample is ranked higher than a randomly chosen negative sample.

---

## 📦 Dataset: Pima Indians Diabetes

We use the classic **Pima Indians Diabetes** dataset:
- **768 patients**, 8 medical features
- **Target:** `Outcome` — 1 = diabetic, 0 = not diabetic
- Features include: Glucose, BMI, Age, Insulin, Blood Pressure, etc.

---
## Step 1 — Load and Explore the Data

We load the dataset directly from GitHub. `data.head()` shows the first 5 rows so we can verify structure.

> **Key columns:** Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age → all used as features.  
> **Target:** `Outcome` (0 = healthy, 1 = diabetic)

In [ ]:
import pandas as pd

# Load the Pima Indians Diabetes dataset from a public GitHub URL
data = pd.read_csv('https://raw.githubusercontent.com/npradaschnor/Pima-Indians-Diabetes-Dataset/master/diabetes.csv')

# Quick sanity check: look at the first 5 rows
print(f"Dataset shape: {data.shape}")
print(f"Class balance:\n{data['Outcome'].value_counts()}")
data.head()

---
## Step 2 — Prepare Features and Target

We split the dataframe into:
- **X** — feature matrix (all columns except `Outcome`)
- **y** — target vector (`Outcome`)

> `axis=1` means we drop a **column** (not a row). The result is a DataFrame of 8 numeric features.

In [ ]:
# X = features (all columns except the target)
X = data.drop('Outcome', axis=1)

# y = target labels (0 or 1)
y = data['Outcome']

print("Feature names:", list(X.columns))
print("X shape:", X.shape)   # (768, 8)
print("y shape:", y.shape)   # (768,)

---
## Step 3 — Train / Test Split

We hold out **20%** of data for evaluation. The model never sees the test set during training.

| Parameter | Value | Meaning |
|---|---|---|
| `test_size=0.2` | 20% test | ~154 samples for testing |
| `random_state=2` | Seed | Ensures reproducibility |

> **Why split?** If we evaluate on training data, the model appears artificially good (overfitting). The test set gives an honest estimate of real-world performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% held out for testing
    random_state=2    # fixed seed for reproducibility
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

---
## Step 4 — Train a Logistic Regression Model

**Logistic Regression** is a linear model that outputs probabilities via the **sigmoid function**:

$$P(y=1) = \frac{1}{1 + e^{-(\mathbf{w} \cdot \mathbf{x} + b)}}$$

- Output is always between 0 and 1 — perfect for probability estimation.
- `max_iter=1000` prevents the solver from stopping too early before converging.

> **Why Logistic Regression for ROC?** It produces **calibrated probabilities**, making it ideal for demonstrating threshold effects.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)  # increase iterations to ensure convergence
model.fit(X_train, y_train)               # train the model on training data

print("Model trained successfully!")
print(f"Training accuracy: {model.score(X_train, y_train):.3f}")
print(f"Test accuracy:     {model.score(X_test, y_test):.3f}")

---
## Step 5 — Get Probability Scores

`predict_proba()` returns a 2-column array:
- Column 0: P(class = 0) — probability of being **not diabetic**
- Column 1: P(class = 1) — probability of being **diabetic**

We slice `[:, 1]` to grab only the **positive class probabilities**. These raw scores (not hard predictions) are used to build the ROC curve.

> **Why use probabilities instead of hard predictions?**  
> Hard predictions (`predict`) are binary — they assume a fixed 0.5 threshold. ROC curves require varying the threshold, so we need the raw score.

In [ ]:
# predict_proba returns shape (n_samples, 2)
# Column 0 = P(not diabetic), Column 1 = P(diabetic)
y_scores = model.predict_proba(X_test)[:, 1]   # keep only P(diabetic)

print(f"Score range: {y_scores.min():.3f} to {y_scores.max():.3f}")
print("\nFirst 10 probability scores:")
print(y_scores[:10].round(3))

---
## Step 6 — Compute the ROC Curve

`roc_curve()` sweeps the threshold from high to low and records TPR and FPR at each step.

It returns three arrays of equal length:

| Array | Meaning |
|---|---|
| `fpr` | False Positive Rate at each threshold |
| `tpr` | True Positive Rate at each threshold |
| `thresholds` | The threshold value used at that point |

> **How is the curve built?**  
> For each unique score value, sklearn uses it as a threshold:  
> samples with score ≥ threshold → predicted Positive.  
> Then TPR and FPR are computed for that threshold. Connecting all these points gives the curve.

> ⚠️ `thresholds[0]` is slightly above the highest score (a sentinel value), so `len(thresholds) = len(fpr) - 1` is **not** a bug.

In [ ]:
from sklearn.metrics import roc_curve

# fpr, tpr, thresholds all have the same length
fpr, tpr, thresholds = roc_curve(y_test, y_scores)

print(f"Number of threshold points: {len(thresholds)}")
print(f"\nFirst 5 thresholds (high → low): {thresholds[:5].round(3)}")
print(f"Last  5 thresholds (near zero):  {thresholds[-5:].round(3)}")
print(f"\nCorresponding FPR at first 5 points: {fpr[:5].round(3)}")
print(f"Corresponding TPR at first 5 points: {tpr[:5].round(3)}")

---
## Step 7 — Plot the Basic ROC Curve (with Threshold Labels)

This plot shows:
1. **The ROC curve** — the path of TPR vs FPR as threshold decreases
2. **Threshold labels** — annotated every 10th point to show which threshold corresponds to each position on the curve
3. **Diagonal baseline** — the random-guessing reference line (AUC = 0.5)

> **How to read this plot:**
> - Moving **up and left** along the curve = good (high TPR, low FPR)
> - Moving **right** = threshold dropped further, catching more positives but also more false alarms
> - A curve that **hugs the top-left corner** → better classifier
> - A curve **on the diagonal** → model is no better than random

> **Note:** We filter every 10th threshold (`n=10`) to avoid label clutter on the plot.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# --- Trace 1: The ROC curve line ---
trace0 = go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines',
    name='ROC curve'
)

# --- Trace 2: Threshold annotations (every 10th point to avoid clutter) ---
n = 10
indices = np.arange(len(thresholds)) % n == 0  # Boolean mask: True every 10th index

trace1 = go.Scatter(
    x=fpr[indices],
    y=tpr[indices],
    mode='markers+text',
    name='Threshold points',
    text=[f"Thr={thr:.2f}" for thr in thresholds[indices]],  # label each dot
    textposition='top center'
)

# --- Trace 3: Diagonal baseline (random classifier) ---
trace2 = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random (Area = 0.5)',
    line=dict(dash='dash')   # dashed line to distinguish from the ROC curve
)

layout = go.Layout(
    title='Receiver Operating Characteristic (with threshold labels)',
    xaxis=dict(title='False Positive Rate (FPR)'),
    yaxis=dict(title='True Positive Rate (TPR)'),
    autosize=False,
    width=800,
    height=800,
    showlegend=False
)

fig = go.Figure(data=[trace0, trace1, trace2], layout=layout)
fig.show()

---
## Step 8 — Find the Optimal Threshold (Youden's J Statistic)

The **optimal threshold** maximises the difference between TPR and FPR.

This is known as **Youden's J statistic**:

$$J = \text{Sensitivity} + \text{Specificity} - 1 = TPR - FPR$$

The threshold at which `J` is maximised is the point on the ROC curve **furthest from the diagonal** — the best balance between catching true positives and avoiding false positives.

> **When should you use a different threshold?**  
> - **Medical screening** (e.g. cancer detection) → prioritise **high TPR**, accept higher FPR  
> - **Spam filters** → prioritise **low FPR**, accept lower TPR  
> - **Youden's J** is a neutral default when both errors are equally costly

In [ ]:
# Youden's J: find index where (TPR - FPR) is maximised
j_scores = tpr - fpr                         # J statistic at each threshold
optimal_idx = np.argmax(j_scores)            # index of maximum J
optimal_threshold = thresholds[optimal_idx]  # the corresponding threshold value

print(f"Optimal threshold (Youden's J): {optimal_threshold:.4f}")
print(f"  → TPR at this point: {tpr[optimal_idx]:.4f}  (sensitivity)")
print(f"  → FPR at this point: {fpr[optimal_idx]:.4f}  (1 - specificity)")
print(f"  → Specificity:       {1 - fpr[optimal_idx]:.4f}")
print(f"  → Youden J value:    {j_scores[optimal_idx]:.4f}")

# Compare with default threshold of 0.5
print(f"\nDefault threshold: 0.5")
print(f"Note: if optimal ≠ 0.5, using the optimal threshold will improve performance.")

---
## Step 9 — Plot ROC Curve with AUC Score

`roc_auc_score()` computes the Area Under the Curve using the **trapezoidal rule** — it sums up thin trapezoids under the ROC curve to estimate the total area.

The AUC is shown in the legend so we can immediately judge model quality.

> **Study note:** The shape of the ROC curve and its AUC do **not** depend on the threshold — they summarise performance across **all** thresholds simultaneously. This is why AUC is preferred over accuracy when comparing models, especially on imbalanced datasets.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from sklearn.metrics import roc_auc_score

# Recompute to be self-contained (fpr, tpr, thresholds already exist from Step 6)
fpr, tpr, thresholds = roc_curve(y_test, y_scores)

# roc_auc_score computes AUC using the trapezoidal rule
roc_auc = roc_auc_score(y_test, y_scores)
print(f"AUC Score: {roc_auc:.4f}")

# ROC curve with AUC in the legend label
trace0 = go.Scatter(
    x=fpr,
    y=tpr,
    mode='lines',
    name=f'ROC curve (AUC = {roc_auc:.2f})'  # AUC shown in legend
)

# Threshold annotation points (every 10th)
n = 10
indices = np.arange(len(thresholds)) % n == 0
trace1 = go.Scatter(
    x=fpr[indices],
    y=tpr[indices],
    mode='markers+text',
    name='Threshold points',
    text=[f"Thr={thr:.2f}" for thr in thresholds[indices]],
    textposition='top center'
)

# Optimal threshold marker
optimal_idx = np.argmax(tpr - fpr)
trace_opt = go.Scatter(
    x=[fpr[optimal_idx]],
    y=[tpr[optimal_idx]],
    mode='markers',
    name=f'Optimal threshold = {thresholds[optimal_idx]:.2f}',
    marker=dict(color='red', size=12, symbol='star')
)

# Diagonal baseline
trace2 = go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    name='Random (AUC = 0.5)',
    line=dict(dash='dash')
)

layout = go.Layout(
    title='ROC Curve with AUC Score',
    xaxis=dict(title='False Positive Rate (FPR)'),
    yaxis=dict(title='True Positive Rate (TPR)'),
    autosize=False,
    width=800,
    height=800,
    showlegend=True
)

fig = go.Figure(data=[trace0, trace1, trace_opt, trace2], layout=layout)
fig.show()

---
## Step 10 — Compare Multiple Models: Logistic Regression vs SVM

One of the most powerful uses of ROC curves is **comparing classifiers** on the same test set.

### Support Vector Machine (SVM) — Quick Recap
- Finds the **maximum-margin hyperplane** between classes.
- `SVC(probability=True)` enables probability output via **Platt scaling** (a calibration step).
- SVMs are sensitive to feature scale → we apply **StandardScaler** first.

### StandardScaler
- Transforms each feature to have **mean = 0, std = 1**.
- **Critical rule:** fit the scaler on **training data only**, then apply the same transformation to test data.
  - `fit_transform(X_train)` → learns mean/std from training set, applies it
  - `transform(X_test)` → applies the **same** learned mean/std (never refits on test data)

> **Why scale for SVM but not for Logistic Regression here?**  
> Logistic Regression can work without scaling (especially with regularisation), but SVM's margin calculation is distance-based and directly affected by feature magnitudes. Scaling is best practice for both, but critical for SVM.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import StandardScaler

# --- Feature Scaling for SVM ---
# IMPORTANT: fit ONLY on training data to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform training data
X_test_scaled  = scaler.transform(X_test)        # transform test data with same params

# --- Model 1: Logistic Regression ---
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)                    # unscaled data is fine for LR
lr_scores = lr_model.predict_proba(X_test)[:, 1]  # P(diabetic)

# --- Model 2: Support Vector Machine ---
# probability=True enables predict_proba() via Platt scaling (slower to train)
svm_model = SVC(probability=True)
svm_model.fit(X_train_scaled, y_train)                    # scaled data for SVM
svm_scores = svm_model.predict_proba(X_test_scaled)[:, 1] # P(diabetic)

# --- Compute ROC curves for both models ---
lr_fpr,  lr_tpr,  lr_thresholds  = roc_curve(y_test, lr_scores)
svm_fpr, svm_tpr, svm_thresholds = roc_curve(y_test, svm_scores)

lr_auc  = roc_auc_score(y_test, lr_scores)
svm_auc = roc_auc_score(y_test, svm_scores)

print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"SVM AUC:                 {svm_auc:.4f}")
print(f"\nBetter model: {'Logistic Regression' if lr_auc >= svm_auc else 'SVM'}")

# --- Plot both ROC curves for comparison ---
trace_lr = go.Scatter(
    x=lr_fpr, y=lr_tpr,
    mode='lines',
    name=f'Logistic Regression (AUC = {lr_auc:.2f})'
)

trace_svm = go.Scatter(
    x=svm_fpr, y=svm_tpr,
    mode='lines',
    name=f'SVM (AUC = {svm_auc:.2f})'
)

trace_diag = go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random (AUC = 0.5)',
    line=dict(dash='dash')
)

layout = go.Layout(
    title='ROC Curve Comparison: Logistic Regression vs SVM',
    xaxis=dict(title='False Positive Rate (FPR)'),
    yaxis=dict(title='True Positive Rate (TPR)'),
    autosize=False,
    width=800,
    height=800,
    showlegend=True
)

fig = go.Figure(data=[trace_lr, trace_svm, trace_diag], layout=layout)
fig.show()

---
## Step 11 — Bonus: Evaluate at the Optimal Threshold

Once you've found the optimal threshold, apply it to generate predictions and compute a **full classification report**.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Find optimal threshold for Logistic Regression using Youden's J
opt_idx = np.argmax(lr_tpr - lr_fpr)
opt_threshold = lr_thresholds[opt_idx]

print(f"Optimal threshold: {opt_threshold:.4f}")

# Apply optimal threshold to convert probabilities to labels
y_pred_default = (lr_scores >= 0.5).astype(int)          # default threshold
y_pred_optimal = (lr_scores >= opt_threshold).astype(int) # optimal threshold

print("\n--- Default Threshold (0.5) ---")
print(classification_report(y_test, y_pred_default, target_names=['Not Diabetic', 'Diabetic']))

print(f"--- Optimal Threshold ({opt_threshold:.3f}) ---")
print(classification_report(y_test, y_pred_optimal, target_names=['Not Diabetic', 'Diabetic']))

---
## 📝 Key Takeaways — Study Summary

### Core Concepts

1. **ROC Curve** = plot of TPR vs FPR at all possible classification thresholds.
2. **Ideal curve** → hugs the top-left corner. **Worst** → along the diagonal.
3. **AUC** = probability that model ranks a random positive higher than a random negative.
4. **Optimal threshold** = argmax(TPR − FPR) — Youden's J statistic.
5. **Model comparison** → plot both ROC curves; higher AUC = better model.

### Common Pitfalls

| Mistake | Correct Approach |
|---|---|
| Using `predict()` for ROC | Use `predict_proba()[:, 1]` |
| Fitting scaler on test data | `fit_transform(train)`, `transform(test)` only |
| Assuming threshold = 0.5 is optimal | Always check with ROC curve |
| Using accuracy on imbalanced data | Use AUC — it is threshold-independent |

### Quick Reference: sklearn Functions

```python
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
auc = roc_auc_score(y_true, y_scores)
optimal_threshold = thresholds[np.argmax(tpr - fpr)]
```

---

### Further Reading
- [scikit-learn ROC documentation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html)
- [ROC and AUC explained visually (Google ML Crash Course)](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc)
- Fawcett, T. (2006). *An introduction to ROC analysis.* Pattern Recognition Letters.